# PTM Corpus Build + ESM-2 Activation Extraction Pipeline
### Salvage-proof: every stage is guarded, long jobs are logged subprocesses, outputs are written as they go

**Modes** (`PTM_SAE_MODE` env var): `smoke` proves the whole notebook on mock data in minutes; `real`
is the full build. **Phase A (this notebook's default job)** builds and finalizes the corpus:
Swiss-Prot human proteome + PTM databases -> CD-HIT@40% clusters -> label cascade -> joint
train/val/held-out split balanced by an optimiser -> `cd-hit-2d` homology audit -> finalized
parquet + manifest. A verification cell gates publishing to the HF corpus repo. **Phase B**
(ESM-2 activation extraction, GPU) is off by default (`RUN_EXTRACTION`).

**Failure handling:** a failing cell is recorded in `run_state.json` and the notebook continues;
dependent cells skip with a reason; the last cell always prints what completed and which files can
be salvaged. Runs identically on a Kaggle kernel and a Linux GPU box (the box is preferred; use
Kaggle only when it is unavailable; a run never uses both).

In [ ]:
# 1. Environment & GPU verification
# Deliberately imports no numpy/torch: dependencies are installed in step 3, and a package imported
# before pip upgrades it stays at the old version inside this kernel (later imports then fail).
import platform
import subprocess

print("python", platform.python_version())
gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "no NVIDIA GPU visible (CPU run)")

In [ ]:
# 2. Repository Setup (same on a local PC, a persistent remote box, Kaggle and Colab)
# Finds the checkout by walking up from the kernel's working directory, so it also works when this
# notebook is opened from notebooks/. Only a runtime with no checkout (fresh Kaggle/Colab) clones;
# an existing checkout is fast-forwarded (local edits are auto-stashed, never discarded) on its
# current branch -- switch branches yourself with `git checkout` in a terminal.
import os
import subprocess
import sys
from pathlib import Path

BRANCH = os.environ.get("PTM_SAE_BRANCH", "main")  # used only when cloning a fresh runtime
PULL = os.environ.get("PTM_SAE_PULL", "1") == "1"  # fast-forward an existing checkout (PTM_SAE_PULL=0 to skip)
REPO = "github.com/Mumu-kun/PTM-SAE.git"

here = Path.cwd().resolve()
root = next(
    (p for p in (here, *here.parents) if (p / "pyproject.toml").is_file() and (p / "src/ptm_sae").is_dir()),
    None,
)

if root is None:
    token = os.environ.get("GH_TOKEN")  # private repo: also read Kaggle/Colab secrets
    try:
        from kaggle_secrets import UserSecretsClient

        token = token or UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata

        token = token or userdata.get("GH_TOKEN")
    except Exception:
        pass

    auth = f"x-access-token:{token}@" if token else ""
    root = (Path("/kaggle/working") if Path("/kaggle").exists() else here) / "PTM-SAE"
    print(f"Cloning {BRANCH} into {root}...")
    try:
        subprocess.run(["git", "clone", "--branch", BRANCH, f"https://{auth}{REPO}", str(root)], check=True, capture_output=True)
    except subprocess.CalledProcessError:
        raise RuntimeError(f"git clone of branch {BRANCH!r} failed: check the branch name and that the GH_TOKEN secret is attached to this notebook") from None
elif PULL:
    print(f"Fast-forwarding existing checkout at {root}...")
    subprocess.run(["git", "pull", "--ff-only", "--autostash"], cwd=root, check=False)

os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from ptm_sae import runtime

print(f"Platform: {runtime.detect_platform()} | repo: {root} | data root: {runtime.resolve_data_root()}")

In [ ]:
# 3. Install Dependencies (single-sourced from pyproject.toml)
# Local/remote boxes manage their environment with `uv sync --extra notebook` in a terminal (locked
# versions, CUDA torch build kept intact), so there this cell only checks it. Kaggle/Colab ship a
# CUDA-matched torch/transformers, so there everything else is pip-installed (retried: a flaky mirror
# must not end a long run) and those two are left alone. A failure is recorded below, not raised.
import importlib.metadata
import subprocess
import sys
import time

EXTRAS = ()
INSTALL_OK = True
# Packages this kernel already imported must not be changed by pip underneath it (the running process
# would be half-old, half-new and later imports fail): remember their versions to check afterwards.
IMPORTED_BEFORE = {n: importlib.metadata.version(n) for n in ("numpy", "pandas", "pyarrow", "scipy", "torch") if n in sys.modules}

if runtime.detect_platform() == "local":
    missing = runtime.missing_requirements(extras=EXTRAS)
    INSTALL_OK = not missing
    if missing:
        print(f"Missing {missing}: run `uv sync --extra notebook` in a terminal, then restart the kernel.")
else:
    requirements = runtime.cloud_requirements(extras=EXTRAS)
    for attempt in range(1, 4):
        deps = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], capture_output=True, text=True)
        local = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps", "--ignore-requires-python"], capture_output=True, text=True)
        if deps.returncode == 0 and local.returncode == 0:
            break
        print(f"pip attempt {attempt} failed: {(deps.stderr + local.stderr)[-400:]}")
        time.sleep(10 * attempt)
    else:
        INSTALL_OK = False
changed = [n for n, version in IMPORTED_BEFORE.items() if importlib.metadata.version(n) != version]
if changed:
    INSTALL_OK = False
    print(f"RESTART THE KERNEL: {changed} were imported before pip changed them; later imports would fail.")
print("dependencies ready:", INSTALL_OK)

In [ ]:
# 4. cd-hit (system package: the corpus build and the homology audit need it)
import shutil
import subprocess

if shutil.which("cd-hit-2d") is None:
    for attempt in range(1, 4):
        result = subprocess.run("apt-get install -y -q cd-hit", shell=True, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"apt-get attempt {attempt} failed: {result.stderr[-300:]}")
print("cd-hit:", shutil.which("cd-hit"), "| cd-hit-2d:", shutil.which("cd-hit-2d"))

In [ ]:
# 5. Run mode, parameters, run state and preflight
import os
import sys
from pathlib import Path

MODE = os.environ.get("PTM_SAE_MODE", "real")  # "smoke": tiny run proving the notebook end to end; "real": the full run
if MODE not in ("smoke", "real"):
    print(f"Unknown PTM_SAE_MODE={MODE!r}; using 'smoke'")
    MODE = "smoke"

FORCE = False  # True -> ignore every cache, rebuild from scratch
PUBLISH = MODE == "real"  # upload the corpus to the HF corpus repo (only after the verification cell passes)
PHASES = os.environ.get("PTM_SAE_PHASES", "corpus")  # "corpus" (CPU) | "extraction" (GPU, needs the published corpus) | "both"
if PHASES not in ("corpus", "extraction", "both"):
    print(f"Unknown PTM_SAE_PHASES={PHASES!r}; using 'corpus'")
    PHASES = "corpus"
RUN_CORPUS = PHASES in ("corpus", "both")
RUN_EXTRACTION = PHASES in ("extraction", "both")

HF_CORPUS_REPO_ID = "mustafa-muhaimin/ptm-sae-corpus"
HF_CORPUS_SUBPATH = "corpus"
KAGGLE_DATASET_SLUG = "mustafamuhaimin/ptm-sae-corpus"  # Kaggle Dataset mirror: attach it to Kaggle runs instead of downloading

DATA_ROOT = runtime.resolve_data_root()
LOG_DIR = DATA_ROOT / "logs"

# Every stage below runs inside `with state.cell(...)`: an exception is recorded and printed, later
# cells that depend on it skip with a reason, and the run carries on, so one silly error can never
# throw away a long run's outputs.
state = runtime.RunState()
with state.cell("install"):
    if not INSTALL_OK:
        raise RuntimeError("dependency install failed (see the install cell output)")

with state.cell("preflight"):
    state.require("install")
    report = runtime.preflight(
        need_gpu=RUN_EXTRACTION, need_cdhit=True, need_hub_write=PUBLISH, min_free_gb=10
    )
    if not report.ok:
        raise RuntimeError(f"preflight failed: {report.failures()}")
    if "publish" in report.degraded:
        PUBLISH = False  # the build still runs and its outputs are kept; publish them afterwards
        print("publishing disabled:", report.degraded["publish"])
print(f"mode={MODE} phases={PHASES} publish={PUBLISH} data root={DATA_ROOT}")

In [ ]:
# 6. Corpus build: acquire -> CD-HIT@40% -> label cascade -> joint split + homology audit -> finalize
# A subprocess, so a crash or out-of-memory kill cannot take the notebook down; progress is mirrored
# here and fully logged. Outputs land in the data root as stages finish, so a late failure keeps them.
with state.cell("corpus"):
    state.require("preflight")
    if not RUN_CORPUS:
        raise runtime.CellSkipped("phases excludes the corpus build")
    cmd = [sys.executable, "-u", "-m", "ptm_sae.corpus.pipeline"]
    cmd += ["--mock"] if MODE == "smoke" else []
    cmd += ["--force"] if FORCE else []
    code = runtime.run_logged(cmd, LOG_DIR / "corpus_build.log", timeout_s=state.time_left())
    if code != 0:
        raise RuntimeError(f"corpus build exited with {code}; see {LOG_DIR / 'corpus_build.log'}")

In [ ]:
# 7. Verify: structure, homology audit clean, balance within tolerance, headline bars met
with state.cell("verify"):
    state.require("corpus")
    from ptm_sae.corpus import pipeline as corpus_pipeline

    problems = corpus_pipeline.verify_outputs(mock=MODE == "smoke")
    for problem in problems:
        print("PROBLEM:", problem)
    if problems:
        raise RuntimeError(f"{len(problems)} verification problem(s): the corpus will NOT be published")
    print("corpus verified")

In [ ]:
# 8. Publish to the HF corpus repo (only after verification), then confirm the upload matches the build
with state.cell("publish"):
    state.require("verify")
    if not PUBLISH:
        raise runtime.CellSkipped("PUBLISH is off")
    from ptm_sae.corpus import pipeline as corpus_pipeline
    from ptm_sae.corpus.config import CorpusPaths
    from ptm_sae.extraction.hub import HfSyncClient

    paths = CorpusPaths.from_env()
    corpus_pipeline.upload_to_hf(paths, HF_CORPUS_REPO_ID, HF_CORPUS_SUBPATH)

    hub = HfSyncClient()
    names = [n for n in corpus_pipeline.ARTIFACT_FILENAMES if (paths.processed_dir / n).exists()]
    remote_paths = {n: hub.build_repo_path(HF_CORPUS_SUBPATH, n) for n in names}
    remote = hub.get_remote_files_info(HF_CORPUS_REPO_ID, list(remote_paths.values()))
    wrong = [n for n in names if remote.get(remote_paths[n], (None,))[0] != (paths.processed_dir / n).stat().st_size]
    if wrong:
        raise RuntimeError(f"published files differ in size from the local build: {wrong}")
    print(f"published and size-verified: {names}")

In [ ]:
# 8b. Kaggle Dataset mirror of the corpus, for attaching to Kaggle runs (needs KAGGLE_USERNAME/KAGGLE_KEY secrets)
with state.cell("publish_kaggle"):
    state.require("publish")
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        value = runtime.resolve_secret(name)
        if not value:
            raise runtime.CellSkipped(f"{name} secret is not attached")
        os.environ[name] = value  # the kaggle package reads its credentials from the environment
    from ptm_sae.corpus.config import CorpusPaths
    from ptm_sae.extraction.hub import publish_kaggle_dataset

    publish_kaggle_dataset(CorpusPaths.from_env().processed_dir, KAGGLE_DATASET_SLUG, "PTM-SAE Corpus", "Corpus rebuild")

In [ ]:
# 9. ESM-2 activation extraction (GPU), resumes from the manifest. Runs only when PHASES includes it.
PROFILE = "PRODUCTION" if MODE == "real" else "SAMPLE"  # SAMPLE (2-minute check) | PILOT (first 500 proteins) | PRODUCTION (full discovery set, 650M)
REMOTE_REPO_ID = "mustafa-muhaimin/ptm-sae-dataset"  # activations repo
PROFILES = {
    "SAMPLE": ("configs/dev_8m.yaml", ["--sample-only"]),
    "PILOT": ("configs/dev_8m.yaml", ["--max-proteins", "500"]),
    "PRODUCTION": ("configs/colab_650m.yaml", []),
}
CONFIG_PATH, EXTRA_ARGS = PROFILES[PROFILE]

with state.cell("extraction"):
    state.require("preflight")
    if not RUN_EXTRACTION:
        raise runtime.CellSkipped("phases excludes extraction")
    if RUN_CORPUS:
        state.require("publish")  # activations must be stamped against the corpus that is on the Hub
    if MODE == "real" and not PUBLISH:
        raise runtime.CellSkipped("no Hub write access (HF_TOKEN missing or read-only): extraction shards would not be kept")
    cmd = [
        sys.executable, "-u", "-m", "ptm_sae.pipeline", "--config", CONFIG_PATH,
        "--remote-repo", REMOTE_REPO_ID, "--remote-corpus-repo", HF_CORPUS_REPO_ID, *EXTRA_ARGS,
    ]
    code = runtime.run_logged(cmd, LOG_DIR / "extraction.log", timeout_s=state.time_left())
    if code != 0:
        raise RuntimeError(f"extraction exited with {code}; see {LOG_DIR / 'extraction.log'} (re-run resumes)")

In [ ]:
# 10. Zero-copy SafeTensors readback sanity check on the extracted shards
with state.cell("readback"):
    state.require("extraction")
    from ptm_sae.config import PipelineConfig
    from ptm_sae.extraction import SafeTensorsReader

    cfg = PipelineConfig.from_yaml(CONFIG_PATH)
    reader = SafeTensorsReader(cache_dir=Path(runtime.anchor_path(cfg.sharding.output_dir, DATA_ROOT)) / "discovery_train")
    proteins = reader.list_proteins()
    print(f"Total accessible proteins: {len(proteins)}")
    if not proteins:
        raise RuntimeError("no proteins readable from the extraction output")
    test_id = proteins[0]
    acts = reader.get_protein_activations(test_id)
    print(f"[Verification] {test_id} tensor {tuple(acts.shape)}, residue 1 {tuple(reader.get_residue_activation(test_id, 1).shape)}")

In [ ]:
# 11. Run summary -- always runs: what completed, what did not, and the files that can be salvaged.
# Depends only on the run state, so it works after any failure above.
try:
    state.print_summary(artifact_dirs=(DATA_ROOT / "data" / "processed", LOG_DIR))
except Exception as exc:  # noqa: BLE001 -- the summary must never be what fails the run
    print(f"summary failed: {type(exc).__name__}: {exc}")